# LegalQA Main 1/3 — QLoRA train / resume / nhập full-run cũ

Add Input: dataset train và ver3-smoke-output. Lần đầu train tối đa 768 QA, 2 epoch. Có output Stage 1 dở dang thì gắn lại làm PREVIOUS_OUTPUT. Khi status=complete, chuyển sang Stage 2.

Để dùng QLoRA từ main-run cũ: đặt LEGACY_INPUT_ROOT tới thư mục legalqa_quality_v8_full đã có config.json, data_public/split_manifest.json và sft (checkpoint epoch 1/2, adapter_last, training_result.json). Notebook kiểm tra dữ liệu/config/model/index, giữ nguồn gốc training cũ, không train lại. Dev và submission sẽ được sinh lại bằng code đã sửa.

Giới hạn: làm việc tối đa 9 giờ tính từ cell đầu, export tối đa 10 phút; supervisor dừng cả nhóm subprocess. Trạng thái paused là kết thúc phiên hợp lệ để Save Output và chạy tiếp. Không cam kết hoàn thành toàn bộ stage trong một phiên. Sự cố hạ tầng Kaggle vẫn có thể làm phiên kết thúc sớm.


In [ ]:
import json, os, signal, subprocess, sys, time
from pathlib import Path

# Count setup/install time too. Do not reset this timestamp in later cells.
SESSION_STARTED = time.monotonic()
if not Path('/kaggle').is_dir():
    raise RuntimeError('Notebook chỉ chạy trên Kaggle.')
WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
REPO_URL = 'https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git'
CODE = WORK / 'legalqa_stage_code'

# 9h includes setup and compute; export gets up to 10 additional minutes.
# The remaining margin is reserved for Kaggle output collection and runtime variation.
WORK_HOURS = 9.0
EXPORT_SECONDS = 600
VERSION3_ROOT = Path('/kaggle/input/datasets/lighth/ver3-smoke-output/legalqa_smoke_full_v1')
DATASET_ROOT = Path('/kaggle/input/datasets/lighth/uit-dsc-2026-task2-legalqa-train')

# None = discover exactly one matching input; set a full ROOT if multiple versions exist.
PREVIOUS_OUTPUT = None   # Cumulative output of THIS stage from an earlier session.
UPSTREAM_OUTPUT = None   # Stage 1 for notebook 02; Stage 2 for notebook 03.
LEGACY_INPUT_ROOT = None # Notebook 01 only: old legalqa_quality_v8_full with completed QLoRA.
REPO_REVISION = None     # First run: main. Continuations: automatically pin upstream commit.

STAGE = 1
MODE = 'auto'
MAX_NEW_QUESTIONS = 200


## Khóa code đúng commit và nhận diện input

Lần đầu dùng main. Các phiên tiếp theo tự checkout full SHA trong manifest, kể cả khi main đã cập nhật. Giữ một output mỗi stage trong Input hoặc đặt ROOT cụ thể.


In [ ]:
if not 0 < WORK_HOURS <= 9:
    raise ValueError('WORK_HOURS phải trong (0, 9]; giữ thời gian dự phòng trước 12h.')
WORK_END = SESSION_STARTED + WORK_HOURS * 3600

def resolve_output(value, stage, required=False):
    marker = f'stage{stage}_manifest.json'
    if value is not None:
        root = Path(value)
        if not (root / marker).is_file():
            raise FileNotFoundError(root / marker)
        return root
    matches = sorted(INPUT.rglob(marker))
    if len(matches) > 1:
        raise RuntimeError(f'Nhiều output Stage {stage}: {matches}. Chỉ định ROOT ở cell cấu hình.')
    if not matches:
        if required:
            raise FileNotFoundError(f'Add Input output Stage {stage} chứa {marker}.')
        return None
    return matches[0].parent

PREVIOUS_OUTPUT = resolve_output(PREVIOUS_OUTPUT, STAGE)
if STAGE > 1:
    UPSTREAM_OUTPUT = resolve_output(UPSTREAM_OUTPUT, STAGE - 1, required=PREVIOUS_OUTPUT is None)
elif UPSTREAM_OUTPUT is not None:
    raise ValueError('Stage 1 không nhận UPSTREAM_OUTPUT.')
if LEGACY_INPUT_ROOT is not None:
    LEGACY_INPUT_ROOT = Path(LEGACY_INPUT_ROOT)
    if STAGE != 1 or PREVIOUS_OUTPUT is not None:
        raise ValueError('Legacy import chỉ dùng ở Stage 1 mới, không trộn với previous output.')

pins = []
for source, number in [(PREVIOUS_OUTPUT, STAGE), (UPSTREAM_OUTPUT, STAGE - 1)]:
    if source is not None:
        info = json.loads((source / f'stage{number}_manifest.json').read_text(encoding='utf-8'))
        if info.get('schema') != 2:
            raise ValueError('Input dùng schema cũ. Chọn đúng output mới hoặc legacy import ở Stage 1.')
        pins.append(info['code_commit'])
if len(set(pins)) > 1:
    raise ValueError('Upstream và previous output khác code commit.')
PIN = pins[0] if pins else (REPO_REVISION or 'main')
if pins and REPO_REVISION and REPO_REVISION != PIN:
    raise ValueError('Không đổi commit khi resume. Bắt đầu một experiment mới nếu cần đổi code.')

class BudgetPause(Exception):
    pass

def bounded_process(command, *, seconds=None, cwd=None, env=None):
    remaining = WORK_END - time.monotonic()
    if remaining <= 0:
        raise BudgetPause('Đã hết ngân sách phiên.')
    limit = remaining if seconds is None else min(remaining, seconds)
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(list(map(str, command)), cwd=cwd, env=env, start_new_session=True)
    try:
        rc = process.wait(timeout=limit)
    except (subprocess.TimeoutExpired, KeyboardInterrupt) as error:
        # Worker and every legalqa subprocess share this process group.
        # Stop all of them before hashing/exporting artifacts.
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            pass
        # The group leader may exit while a GPU child ignores SIGTERM.
        # Always kill remaining group members before exporting.
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait(timeout=30)
        if isinstance(error, KeyboardInterrupt):
            raise
        raise BudgetPause('Đã dừng worker theo ngân sách; tiến độ đã ghi sẽ được export.') from error
    if rc:
        raise subprocess.CalledProcessError(rc, command)

if CODE.exists():
    if not (CODE / '.git').is_dir():
        raise RuntimeError(f'{CODE} không phải repo. Dùng phiên Kaggle mới.')
    remote = subprocess.check_output(['git', '-C', str(CODE), 'remote', 'get-url', 'origin'], text=True, timeout=30).strip()
    if remote.rstrip('/') != REPO_URL.rstrip('/'):
        raise RuntimeError('Repo origin không khớp.')
    dirty = subprocess.check_output(['git', '-C', str(CODE), 'status', '--porcelain'], text=True, timeout=30).strip()
    if dirty:
        raise RuntimeError('Code trong session có sửa đổi; không tự ghi đè. Dùng phiên mới.')
else:
    bounded_process(['git', 'clone', '--no-checkout', '--depth', '1', REPO_URL, CODE], seconds=300)
bounded_process(['git', '-C', CODE, 'fetch', '--depth', '1', 'origin', PIN], seconds=300)
bounded_process(['git', '-C', CODE, 'checkout', '--detach', 'FETCH_HEAD'], seconds=60)
commit = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True, timeout=30).strip()
if pins and commit != PIN:
    raise RuntimeError('Checkout không đúng commit đã khóa.')
if not (CODE / 'legalqa' / 'stages.py').is_file():
    raise RuntimeError('Commit chưa có stages.py. Push các thay đổi mới trước khi chạy.')
print('Pinned commit:', commit)
print('Previous:', PREVIOUS_OUTPUT, '| Upstream:', UPSTREAM_OUTPUT)


## Cài môi trường và kiểm tra


In [ ]:
bounded_process([sys.executable, '-m', 'pip', 'install', '-q', '-r', CODE / 'requirements.txt'], seconds=1200)
if STAGE == 2:
    bounded_process([sys.executable, '-m', 'nltk.downloader', '-q', 'wordnet', 'omw-1.4'], seconds=300)
    bounded_process([sys.executable, 'scripts/check_metrics.py'], cwd=CODE, seconds=300)
bounded_process([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=CODE, seconds=300)


## Chạy trong ngân sách và export cả tiến độ dở dang

Mọi xử lý/kiểm tra artifact dùng legalqa/stages.py chung cho ba notebook. Diagnostics chứa dữ liệu dev, reference, retrieval, prediction, audit, metrics và trạng thái train đã có; không chứa trọng số.


In [ ]:
RUN_ROOT = WORK / f'legalqa_main_stage{STAGE}_v8'
OPTIONS = WORK / f'legalqa_stage{STAGE}_options.json'
options = {
    'stage': STAGE, 'root': str(RUN_ROOT), 'version3': str(VERSION3_ROOT),
    'dataset': str(DATASET_ROOT), 'mode': MODE, 'max_new_questions': MAX_NEW_QUESTIONS,
    'previous': str(PREVIOUS_OUTPUT) if PREVIOUS_OUTPUT is not None else None,
    'upstream': str(UPSTREAM_OUTPUT) if UPSTREAM_OUTPUT is not None else None,
    'legacy': str(LEGACY_INPUT_ROOT) if LEGACY_INPUT_ROOT is not None else None,
}
OPTIONS.write_text(json.dumps(options, ensure_ascii=False, indent=2), encoding='utf-8')
# Cooperative pause 10 minutes before hard worker stop, for saving Trainer state.
remaining = max(0, WORK_END - time.monotonic())
worker_env = {**os.environ, 'LEGALQA_DEADLINE': str(time.time() + max(0, remaining - 600)),
              'PYTHONUNBUFFERED': '1'}
outcome, failure = 'ok', None
try:
    bounded_process([sys.executable, '-m', 'legalqa.stages', 'run', '--options', OPTIONS],
                    cwd=CODE, env=worker_env)
except BudgetPause as error:
    outcome = 'paused'
    print(str(error), flush=True)
except Exception as error:
    outcome, failure = 'failed', error
finally:
    if (RUN_ROOT / 'session.json').is_file():
        # Export runs only after the entire compute process group has stopped.
        # It is bounded separately, without restarting the 9h compute budget.
        original_end = WORK_END
        WORK_END = min(SESSION_STARTED + 10 * 3600, time.monotonic() + EXPORT_SECONDS)
        try:
            bounded_process([sys.executable, '-m', 'legalqa.stages', 'finalize',
                             '--options', OPTIONS, '--outcome', outcome], cwd=CODE)
        finally:
            WORK_END = original_end
if failure is not None:
    raise failure
manifest_path = RUN_ROOT / f'stage{STAGE}_manifest.json'
if not manifest_path.is_file():
    raise RuntimeError('Chưa tạo được snapshot; xem lỗi setup ở trên.')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print('STATUS:', manifest['status'])
print('PROGRESS:', manifest['progress'])
print('OUTPUT:', RUN_ROOT)
if manifest['status'] == 'complete':
    print('Stage hoàn tất. Có thể dùng output cho stage tiếp theo.')
else:
    print('Phiên kết thúc có chủ đích. Save output, Add Input vào CÙNG notebook, rồi chạy tiếp.')
print('submission.zip chỉ được tạo khi đủ tất cả ID public, không nộp partial JSON.')
